# Step 05.3 — Gradient 与 Automatic Differentiation

5.2 里只有一个 parameter `w`，所以我们只需要一个导数 `dL/dw`。

真实神经网络有很多 parameter。于是问题变成：

> loss 对每一个 parameter 的导数分别是多少？

把这些导数收集起来，就是 gradient（梯度）。

## 1. 从一个导数扩展到 Gradient

做一个只有两个 parameter 的模型：

```text
prediction = w0*x0 + w1*x1
```

设：

```text
w = [3, 1]
x = [2, -1]
target = 10
```

prediction：

```text
3*2 + 1*(-1) = 5
```

error：

```text
5 - 10 = -5
```

loss：

```text
(-5)^2 = 25
```

现在我们需要两个导数：

```text
dL/dw0
dL/dw1
```

把它们排成和 w 一样的结构：

```text
gradient = [dL/dw0, dL/dw1]
```

In [ ]:
import mlx.core as mx

w = mx.array([3.0, 1.0])
x = mx.array([2.0, -1.0])
target = mx.array(10.0)

prediction = mx.sum(w * x)
error = prediction - target
loss = mx.square(error)

mx.eval(prediction, error, loss)

print("prediction:", prediction)
print("error     :", error)
print("loss      :", loss)

### 手算两个导数

因为：

```text
L = (w0*x0 + w1*x1 - target)^2
```

令：

```text
e = prediction - target
```

则：

```text
dL/de = 2e
de/dw0 = x0
de/dw1 = x1
```

所以：

```text
dL/dw0 = 2e*x0
dL/dw1 = 2e*x1
```

代入 `e=-5, x=[2,-1]`：

```text
dL/dw0 = 2*(-5)*2    = -20
dL/dw1 = 2*(-5)*(-1)= 10
```

所以：

```text
gradient = [-20, 10]
```

In [ ]:
manual_gradient = 2 * error * x
mx.eval(manual_gradient)

print("manual gradient:", manual_gradient)

这里 gradient 的 shape 和 parameter `w` 一样：

```text
w.shape        = (2,)
gradient.shape = (2,)
```

每一个位置都回答：

> 当前这个 parameter 稍微变化，会让 loss 变化多快？

## 2. Gradient 的方向是什么意思？

gradient 指向 loss 局部增长最快的方向。

当前：

```text
gradient = [-20, 10]
```

所以：

- `dL/dw0 = -20`：增大 `w0` 会让 loss 局部下降；
- `dL/dw1 = +10`：增大 `w1` 会让 loss 局部上升，因此若想减小 loss，应该倾向于减小 `w1`。

这就是为什么后面 gradient descent 会沿着：

```text
-gradient
```

走，而不是沿 gradient 走。

真正的参数更新放到 5.4 再做。

## 3. 先用 finite difference 验证每个 parameter

5.2 的有限差分可以逐个 parameter 使用。

只扰动 `w0`，估计 `dL/dw0`；再只扰动 `w1`，估计 `dL/dw1`。

In [ ]:
def loss_fn_python(w0, w1):
    x0, x1 = 2.0, -1.0
    target = 10.0
    prediction = w0 * x0 + w1 * x1
    return (prediction - target) ** 2

w0, w1 = 3.0, 1.0
h = 1e-4

d_w0 = (loss_fn_python(w0 + h, w1) - loss_fn_python(w0, w1)) / h
d_w1 = (loss_fn_python(w0, w1 + h) - loss_fn_python(w0, w1)) / h

print("finite diff dL/dw0:", d_w0)
print("finite diff dL/dw1:", d_w1)

结果应该接近：

```text
-20
 10
```

但是如果模型有一百万个 parameter，就不能真的把每个 parameter 都 `+h` 跑一次 forward。

这就是 automatic differentiation（自动微分）存在的原因。

## 4. Computation Graph 与 Reverse-Mode Intuition

我们的计算可以画成：

```text
w0 ----\
        * x0 --\
               + --> prediction --> error --> square --> loss
w1 ----\       /
        * x1 --
```

forward 时，我们从左往右得到 loss。

求导时，可以从 scalar loss 反向走：

```text
loss
 ↓
square
 ↓
error
 ↓
prediction
 ↓        ↓
w0       w1
```

每经过一个操作，就使用它自己的局部导数和 chain rule，把 loss 的敏感度继续往前传。

这就是 reverse-mode automatic differentiation 的核心直觉，也是 backpropagation 的数学骨架。

重要：它不是 finite difference，也不是把 parameter 一个个扰动。

## 5. MLX 的 `mx.grad`

先把 loss 写成一个函数：

```python
def loss_fn(w):
    ...
    return scalar_loss
```

然后：

```python
grad_fn = mx.grad(loss_fn)
```

注意：这句不是“现在就求一次 gradient”。

它会产生一个新的函数 `grad_fn`，这个新函数输入 `w`，输出 `loss_fn` 对 `w` 的 gradient。

In [ ]:
x = mx.array([2.0, -1.0])
target = mx.array(10.0)

def loss_fn(w):
    prediction = mx.sum(w * x)
    return mx.square(prediction - target)

w = mx.array([3.0, 1.0])

grad_fn = mx.grad(loss_fn)
gradient = grad_fn(w)

mx.eval(gradient)

print("MLX gradient:", gradient)
print("shape       :", gradient.shape)

应该得到接近：

```text
[-20, 10]
```

也就是和手算、finite difference 一致。

## 6. `mx.value_and_grad`：一次同时拿到 loss 和 gradient

训练时通常两个都要：

- loss：看模型当前错多少；
- gradient：知道 parameter 应该怎么改。

因此 MLX 还提供：

```python
mx.value_and_grad(loss_fn)
```

它同样返回一个新函数。调用这个新函数时，同时得到 value 和 gradient。

In [ ]:
loss_and_grad_fn = mx.value_and_grad(loss_fn)

loss_value, gradient = loss_and_grad_fn(w)
mx.eval(loss_value, gradient)

print("loss    :", loss_value)
print("gradient:", gradient)

现在我们已经从：

```text
parameter
   ↓
prediction
   ↓
loss
```

得到了反方向的信息：

```text
loss
 ↓ automatic differentiation
gradient for every parameter
```

### 5.3 最终结论

- 一个 parameter 对应一个偏导数；
- 所有 parameter 的偏导数组合起来就是 gradient；
- gradient 与 parameter 结构对应；
- gradient 指向 loss 局部增长最快方向；
- reverse-mode autodiff 从 scalar loss 向前面的 parameter 反向传播敏感度；
- `mx.grad(loss_fn)` 生成“求 gradient 的函数”；
- `mx.value_and_grad(loss_fn)` 生成“同时求 loss 和 gradient 的函数”；
- 下一节 5.4：真正使用 gradient 修改 parameter，并亲眼看到 loss 一步步下降。